In [72]:
import pandas as pd
from tqdm import tqdm
import numpy as np

In [2]:
metadata = pd.read_csv("/home/luna.kuleuven.be/u0169940/Downloads/metadata.tsv", sep="\t")

In [47]:
s3_links = metadata[metadata['File analysis status']=="released"]["s3_uri"].to_list()

In [77]:
np.where(metadata["File type"]=="fastq")[0][:5]

array([ 0,  1, 20, 21, 40])

In [50]:
metadata[metadata['File analysis status']=="released"]["File type"].value_counts()

File type
bed       489
bigBed    489
bigWig    485
bam       163
Name: count, dtype: int64

In [43]:
metadata["File type"].value_counts()

File type
fastq     1090
bigBed     882
bed        882
bigWig     616
bam        364
Name: count, dtype: int64

In [4]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config


# Configure S3 client for public (unsigned) access
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
sizes = []
for link in tqdm(s3_links):
    # Parse the S3 URI (s3://bucket/key)
    path_parts = link.replace("s3://", "").split("/", 1)
    bucket = path_parts[0]
    key = path_parts[1]
    
    try:
        # head_object retrieves metadata only (no download)
        response = s3.head_object(Bucket=bucket, Key=key)
        size_bytes = response['ContentLength']
        size_gb = size_bytes / (1024**3)
        sizes.append((link,size_gb))
        # print(f"{key.split('/')[-1]}: {size_gb:.2f} GB")
        
    except Exception as e:
        print(f"Error accessing {link}: {e}")

100%|██████████| 1626/1626 [04:27<00:00,  6.07it/s]


In [5]:
links = metadata[(metadata['File analysis status']=="released") & (metadata['File type']=="bam")]["s3_uri"][:5].to_list()

In [8]:
sizes_pd = pd.DataFrame(sizes, columns=["File", "Size"])

In [26]:
sizes_pd[sizes_pd["File"].apply(lambda x: "bam" in x)].sort_values("Size")

,File,Size
496,s3://encode-public/2022/01/18/c871113a-efdf-45...,4.567707
1326,s3://encode-public/2021/03/09/7d2d3d5a-223e-4a...,10.118629
537,s3://encode-public/2020/10/14/8b4d0c9e-f488-42...,10.455732
1402,s3://encode-public/2020/10/14/f75ab6f3-f799-41...,10.628567
1136,s3://encode-public/2020/10/13/af3e46c0-14a5-48...,11.125509
...,...,...
241,s3://encode-public/2022/01/10/42de7347-233b-47...,233.554499
640,s3://encode-public/2022/01/18/c6f6e903-ab94-4a...,240.749600
895,s3://encode-public/2022/02/16/22fce4f3-764f-41...,257.326755
760,s3://encode-public/2022/01/19/cc83009f-6ab6-44...,357.216724


### Performance assesment ###

In [2]:
import json
import os

In [24]:
experiments = [x for x in os.listdir("../Tutorials") if "DMRs_stratified" in x]

In [26]:
experiment_results = []
for experiment in experiments:
    file_folder = "../Tutorials/"+experiment
    checkpoint = os.listdir(file_folder)[-1]
    trainer_state_path = os.path.join(file_folder,checkpoint ,"trainer_state.json")
    with open(trainer_state_path, "rb") as f:
        log = json.load(f)
    best_step = log["best_global_step"]
    metrics = [x for x in log['log_history'] if x["step"] == best_step][1]
    experiment_results.append(
        {"experiment":experiment,
         'eval_accuracy': round(metrics["eval_accuracy"],4),
          'eval_f1': round(metrics["eval_f1"],4),
          'eval_loss': round(metrics["eval_loss"],4),
          'eval_matthews_correlation': round(metrics["eval_matthews_correlation"],4),
          'eval_precision': round(metrics["eval_precision"],4),
          'eval_recall': round(metrics["eval_recall"],4)
         }
    )


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re

def simplify_experiment_name(name):
    """Extract meaningful components from experiment name."""
    # Remove common prefix
    base = "methylBertLoyferWithReject_attentionClassifier_DMRs_stratified"
    suffix = name.replace(base, "").strip("_")
    
    if not suffix:
        return "baseline"
    
    # Parse components
    parts = []
    
    # Genome version
    if "hg19" in suffix:
        parts.append("hg19")
    elif "hg38" in suffix:
        parts.append("hg38")
    
    # Label type
    if "dmr_ctype_label" in suffix:
        parts.append("ctype")
    elif "dmr_label" in suffix:
        parts.append("dmr")
    
    # Min CpG parameter
    mincpg_match = re.search(r'mincpg_(\d+)', suffix)
    if mincpg_match:
        parts.append(f"cpg≥{mincpg_match.group(1)}")
    
    # Version or split info
    if "original_split" in suffix:
        parts.append("orig_split")
    elif "_v2" in suffix:
        parts.append("v2")
    
    return " | ".join(parts) if parts else "baseline"


def create_visualizations(data):
    """Create comprehensive visualizations for experiment results."""
    
    # Create DataFrame
    df = pd.DataFrame(data)
    df['short_name'] = df['experiment'].apply(simplify_experiment_name)
    
    # Sort by accuracy for better visualization
    df = df.sort_values('eval_accuracy', ascending=True)
    
    # Define metrics (excluding loss for the main comparison)
    performance_metrics = ['eval_accuracy', 'eval_f1', 'eval_precision', 
                          'eval_recall', 'eval_matthews_correlation']
    metric_labels = ['Accuracy', 'F1 Score', 'Precision', 'Recall', 'MCC']
    
    # Color palette
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(performance_metrics)))
    
    # Create figure with subplots
    fig = plt.figure(figsize=(16, 14))
    
    # =========================================================================
    # Plot 1: Grouped Bar Chart - All Metrics Comparison
    # =========================================================================
    ax1 = fig.add_subplot(2, 2, 1)
    
    x = np.arange(len(df))
    width = 0.15
    
    for i, (metric, label) in enumerate(zip(performance_metrics, metric_labels)):
        offset = (i - len(performance_metrics)/2 + 0.5) * width
        bars = ax1.barh(x + offset, df[metric], width, label=label, color=colors[i])
    
    ax1.set_yticks(x)
    ax1.set_yticklabels(df['short_name'], fontsize=9)
    ax1.set_xlabel('Score', fontsize=11)
    ax1.set_title('Performance Metrics Comparison', fontsize=13, fontweight='bold')
    ax1.legend(loc='lower right', fontsize=9)
    ax1.set_xlim(0.7, 1.0)
    ax1.axvline(x=0.9, color='gray', linestyle='--', alpha=0.5, label='0.9 threshold')
    ax1.grid(axis='x', alpha=0.3)
    
    # =========================================================================
    # Plot 2: Accuracy vs Loss (Trade-off visualization)
    # =========================================================================
    ax2 = fig.add_subplot(2, 2, 2)
    
    scatter = ax2.scatter(df['eval_loss'], df['eval_accuracy'], 
                         c=df['eval_f1'], cmap='RdYlGn', s=150, 
                         edgecolors='black', linewidth=1)
    
    # Add labels
    for idx, row in df.iterrows():
        ax2.annotate(row['short_name'], (row['eval_loss'], row['eval_accuracy']),
                    fontsize=7, ha='center', va='bottom', 
                    xytext=(0, 5), textcoords='offset points', rotation=15)
    
    cbar = plt.colorbar(scatter, ax=ax2)
    cbar.set_label('F1 Score', fontsize=10)
    ax2.set_xlabel('Loss (lower is better)', fontsize=11)
    ax2.set_ylabel('Accuracy (higher is better)', fontsize=11)
    ax2.set_title('Accuracy vs Loss Trade-off\n(color = F1 Score)', fontsize=13, fontweight='bold')
    ax2.grid(alpha=0.3)
    
    # =========================================================================
    # Plot 3: Heatmap of all metrics
    # =========================================================================
    ax3 = fig.add_subplot(2, 2, 3)
    
    # Prepare heatmap data (normalize loss by inverting for consistent color scale)
    heatmap_data = df[performance_metrics].copy()
    heatmap_data.index = df['short_name']
    
    im = ax3.imshow(heatmap_data.values, cmap='RdYlGn', aspect='auto', 
                    vmin=0.75, vmax=0.95)
    
    ax3.set_xticks(np.arange(len(metric_labels)))
    ax3.set_xticklabels(metric_labels, fontsize=10, rotation=45, ha='right')
    ax3.set_yticks(np.arange(len(df)))
    ax3.set_yticklabels(df['short_name'], fontsize=9)
    
    # Add value annotations
    for i in range(len(df)):
        for j in range(len(performance_metrics)):
            value = heatmap_data.iloc[i, j]
            text_color = 'white' if value < 0.85 else 'black'
            ax3.text(j, i, f'{value:.3f}', ha='center', va='center', 
                    fontsize=8, color=text_color)
    
    plt.colorbar(im, ax=ax3, label='Score')
    ax3.set_title('Performance Heatmap', fontsize=13, fontweight='bold')
    
    # =========================================================================
    # Plot 4: Radar Chart for Top 5 Experiments
    # =========================================================================
    ax4 = fig.add_subplot(2, 2, 4, projection='polar')
    
    top5 = df.nlargest(5, 'eval_accuracy')
    
    angles = np.linspace(0, 2*np.pi, len(performance_metrics), endpoint=False).tolist()
    angles += angles[:1]  # Complete the circle
    
    colors_radar = plt.cm.Set2(np.linspace(0, 1, 5))
    
    for idx, (_, row) in enumerate(top5.iterrows()):
        values = [row[m] for m in performance_metrics]
        values += values[:1]  # Complete the circle
        ax4.plot(angles, values, 'o-', linewidth=2, label=row['short_name'], 
                color=colors_radar[idx])
        ax4.fill(angles, values, alpha=0.1, color=colors_radar[idx])
    
    ax4.set_xticks(angles[:-1])
    ax4.set_xticklabels(metric_labels, fontsize=9)
    ax4.set_ylim(0.8, 0.95)
    ax4.set_title('Top 5 Experiments\n(Radar Comparison)', fontsize=13, fontweight='bold', y=1.08)
    ax4.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=8)
    
    plt.tight_layout()
    plt
    # plt.savefig('/home/claude/experiment_results.png', dpi=150, bbox_inches='tight', 
    #             facecolor='white', edgecolor='none')
    plt.close()
    
    # =========================================================================
    # Create Summary Table
    # =========================================================================
    print("\n" + "="*80)
    print("EXPERIMENT RESULTS SUMMARY (Sorted by Accuracy)")
    print("="*80)
    
    summary_df = df[['short_name', 'eval_accuracy', 'eval_f1', 'eval_precision', 
                     'eval_recall', 'eval_matthews_correlation', 'eval_loss']].copy()
    summary_df.columns = ['Experiment', 'Accuracy', 'F1', 'Precision', 'Recall', 'MCC', 'Loss']
    summary_df = summary_df.sort_values('Accuracy', ascending=False)
    
    print(summary_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    
    print("\n" + "="*80)
    print("KEY INSIGHTS")
    print("="*80)
    
    best = df.loc[df['eval_accuracy'].idxmax()]
    worst = df.loc[df['eval_accuracy'].idxmin()]
    
    print(f"\n🏆 Best Performer: {best['short_name']}")
    print(f"   Accuracy: {best['eval_accuracy']:.4f} | F1: {best['eval_f1']:.4f} | Loss: {best['eval_loss']:.4f}")
    
    print(f"\n📉 Lowest Performer: {worst['short_name']}")
    print(f"   Accuracy: {worst['eval_accuracy']:.4f} | F1: {worst['eval_f1']:.4f} | Loss: {worst['eval_loss']:.4f}")
    
    print(f"\n📊 Performance Range:")
    print(f"   Accuracy: {df['eval_accuracy'].min():.4f} - {df['eval_accuracy'].max():.4f} (Δ={df['eval_accuracy'].max()-df['eval_accuracy'].min():.4f})")
    print(f"   F1 Score: {df['eval_f1'].min():.4f} - {df['eval_f1'].max():.4f} (Δ={df['eval_f1'].max()-df['eval_f1'].min():.4f})")
    
    # Effect of min CpG
    cpg_experiments = df[df['short_name'].str.contains('cpg≥')]
    if len(cpg_experiments) > 0:
        print(f"\n🧬 Effect of Min CpG Threshold:")
        for _, row in cpg_experiments.sort_values('short_name').iterrows():
            print(f"   {row['short_name']}: Acc={row['eval_accuracy']:.4f}, F1={row['eval_f1']:.4f}")


In [32]:
create_visualizations(experiment_results)


EXPERIMENT RESULTS SUMMARY (Sorted by Accuracy)
               Experiment  Accuracy     F1  Precision  Recall    MCC   Loss
       hg38 | dmr | cpg≥4    0.9378 0.8904     0.9158  0.8727 0.9161 0.1752
     hg38 | ctype | cpg≥4    0.9375 0.8900     0.9115  0.8755 0.9157 0.1755
       hg38 | dmr | cpg≥3    0.9290 0.8895     0.9289  0.8643 0.9037 0.2019
     hg38 | ctype | cpg≥3    0.9286 0.8833     0.9331  0.8523 0.9030 0.2024
                 baseline    0.9200 0.8511     0.8792  0.8280 0.8909 0.2278
hg19 | ctype | orig_split    0.9195 0.8503     0.8604  0.8441 0.8908 0.2222
       hg38 | dmr | cpg≥2    0.9183 0.8740     0.8930  0.8606 0.8893 0.2212
                 baseline    0.9171 0.8490     0.8605  0.8412 0.8872 0.2256
             hg38 | ctype    0.9147 0.8428     0.8621  0.8272 0.8841 0.2308
     hg38 | ctype | cpg≥2    0.9127 0.8688     0.8760  0.8645 0.8820 0.2329
                     hg38    0.9080 0.8258     0.8563  0.8050 0.8742 0.2779
             hg19 | ctype    0.9039 0.8